## Image segmentation with SAM 3

This notebook demonstrates how to use SAM 3 for image segmentation with text or visual prompts. It covers the following capabilities:

- **Text prompts**: Using natural language descriptions to segment objects (e.g., "person", "face")
- **Box prompts**: Using bounding boxes as exemplar visual prompts

In [ ]:
import os
import gc
import sys
import cv2
sys.path.insert(0, "/home/groups/sammer/haogeh/util/models/sam3/")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
import sam3
from PIL import Image
from sam3 import build_sam3_image_model
from sam3.model.box_ops import box_xywh_to_cxcywh
from sam3.model.sam3_image_processor import Sam3Processor
from sam3.visualization_utils import draw_box_on_image, normalize_bbox, plot_results

sam3_root = os.path.join(os.path.dirname(sam3.__file__), "..")

In [ ]:
import importlib
importlib.reload(sam3)

In [ ]:
import torch

# turn on tfloat32 for Ampere GPUs
# https://pytorch.org/docs/stable/notes/cuda.html#tensorfloat-32-tf32-on-ampere-devices
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# use bfloat16 for the entire notebook
torch.autocast("cuda", dtype=torch.bfloat16).__enter__()

# Build Model

In [ ]:
bpe_path = f"{sam3_root}/assets/bpe_simple_vocab_16e6.txt.gz"
model = build_sam3_image_model(
    bpe_path=bpe_path,
    checkpoint_path = f'{sam3_root}/assets/checkpoint/sam3.pt',
    enable_inst_interactivity=True)

In [ ]:
dataset_path = os.path.join(sam3_root,'assets/images/dataset_basal_view_consented')
bbox_file = os.path.join(dataset_path,"df_with_nostril_verified.pkl")

In [ ]:
bbox_df = pd.read_pickle(bbox_file)

In [ ]:
def load_mask_dict(mask_path):
    mask_npz = np.load(mask_path,allow_pickle=True)
    mask_dict = {}
    mask_dict['bbox_xyxy'] = mask_npz['bbox_xyxy']
    mask_dict['bbox_xywh'] = mask_npz['bbox_xywh']
    mask_shape = mask_npz['shape']
    mask_dict['mask'] = np.unpackbits(mask_npz['mask'])[:np.prod(mask_shape)].reshape(mask_shape).astype(bool)
    mask_dict['mask_logit'] = mask_npz['mask_logit']
    mask_dict['score'] = mask_npz['score']
    return mask_dict

In [ ]:
def xyxy_to_xywh(bbox):
    """
    Convert a bounding box from (x1, y1, x2, y2) format to (x, y, w, h) format.
    (x, y) is the top-left corner, (w, h) is width and height.
    """
    x1, y1, x2, y2 = bbox
    x = x1
    y = y1
    w = x2 - x1
    h = y2 - y1
    return (x, y, w, h)

In [ ]:
from skimage.morphology import remove_small_objects
def extract_nose_image(image,mask,x1,y1,x2,y2):
    left, top, right, bottom = int(x1), int(y1), int(x2), int(y2)

    nose_image_raw = image.crop((left, top, right, bottom))
    mask_crop = mask[top:bottom,left:right]

    # Crop nose
    white_bg = Image.new("RGB", nose_image_raw.size, (255, 255, 255))
    mask_pil = Image.fromarray(mask_crop.astype('uint8') * 255)
    nose_image_masked = Image.composite(nose_image_raw, white_bg, mask_pil)
    return nose_image_raw, nose_image_masked
def get_nostril_mask(nose_image):
    lower = 0
    upper = np.percentile(nose_image,10)
    nose_image_np = np.array(nose_image)
    nose_image_gray = cv2.cvtColor(nose_image_np,cv2.COLOR_RGB2GRAY)
    mask = cv2.inRange(nose_image_gray,lower,upper)

    kernel = np.ones((5,5), np.uint8)
    mask = cv2.erode(mask, kernel, iterations=1)
    mask = cv2.dilate(mask, kernel, iterations=1)
    
    mask = (mask > 0)
    mask = remove_small_objects(mask, min_size=100)
    mask = (mask.astype(np.uint8) * 255)

    # Find connected components and their areas
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(mask.astype(np.uint8))
    
    # Get areas for each component (excluding background which is label 0)
    areas = stats[1:, cv2.CC_STAT_AREA]
    
    
    # Get indices of top 2 largest components
    if len(areas) >= 2:
        top_2_indices = np.argsort(-areas)[:2] + 1  # Add 1 since we excluded background
        components = [labels == i for i in top_2_indices]
        medoids = []
        for component in components:
            ys, xs = np.nonzero(component)
            cx, cy = xs.mean(), ys.mean()          # centroid (float)
            pts = np.column_stack([xs, ys]).astype(np.float32)
            d2 = (pts[:,0]-cx)**2 + (pts[:,1]-cy)**2
            px, py = pts[np.argmin(d2)].astype(int)
            medoids.append((px, py))
        medoids = np.array(medoids)
        # Sort by x-coordinate to get left and right
        r_nostril_medoid, l_nostril_medoid = sorted(medoids, key=lambda x: x[0])

        mask = np.isin(labels, top_2_indices).astype(np.uint8) * 255
    else:
        print(f"ERROR: Only {len(areas)} components found in {image_path}")
        return mask, None, None

    return mask, l_nostril_medoid, r_nostril_medoid
def normalize_point(point,image):
    x,y = point
    width,height = image.size
    return x/width,y/height

In [ ]:
# for i,line in bbox_df.sample(5).iterrows():
#     image_path = os.path.join(dataset_path,line['save_path_rel'])
#     result_image_path = os.path.join(os.path.dirname(image_path), os.path.basename(image_path).split('.')[0] + '_nostril_result.jpg')
#     mask_path = os.path.join(os.path.dirname(image_path), os.path.basename(image_path).split('.')[0] + '_mask.npz')
#     # load: state = np.load("state.npz", allow_pickle=True)

#     image = Image.open(image_path)
#     mask_dict = load_mask_dict(mask_path)
#     nose_mask = mask_dict['mask']

#     width, height = image.size
    

#     bbox = line['bbox']
#     x1, y1, x2, y2 = bbox
#     x,y,w,h = xyxy_to_xywh(bbox)

#     box_input_xywh = torch.tensor([x,y,w,h]).view(-1, 4)
#     box_input_cxcywh = box_xywh_to_cxcywh(box_input_xywh)
#     norm_box_cxcywh = normalize_bbox(box_input_cxcywh, width, height).flatten().tolist()

#     nose_image_raw, nose_image_masked= extract_nose_image(image,nose_mask,x1,y1,x2,y2)
#     nostril_mask, l_nostril_medoid, r_nostril_medoid = get_nostril_mask(nose_image_raw)
#     # l_nostril_medoid_norm = normalize_point(l_nostril_medoid,nose_image_raw)
#     # r_nostril_medoid_norm = normalize_point(r_nostril_medoid,nose_image_raw)

#     # points = torch.tensor([l_nostril_medoid_norm], dtype=torch.float32, device=model.device).unsqueeze(0)
#     # labels = torch.tensor([[1]], dtype=torch.int32, device=model.device)

#     inference_state = processor.set_image(nose_image_raw)
#     processor.reset_all_prompts(inference_state)
#     inference_state = processor.set_text_prompt(state=inference_state, prompt="right nostril from basal view")
#     # inference_state = processor.add_geo(
#     #     state=inference_state,
#     #     point_coords=points,
#     #     point_labels=labels
#     # )

#     # mask = inference_state['masks'].cpu().numpy()[0,0]
#     # mask_logit = inference_state['masks_logits'].cpu().numpy()[0,0]
#     # score = inference_state['scores'].float().cpu().numpy()[0]
#     point_coords = [l_nostril_medoid]
#     point_labels = [1] 
#     masks, scores, logits = model.predict_inst(
#         inference_state,
#         point_coords=point_coords,
#         point_labels=point_labels,
#         multimask_output=False,
#     )

#     mask = masks[0]

#     save_dict = {}
#     save_dict['bbox_xyxy'] = x1,y1,x2,y2
#     save_dict['bbox_xywh'] = x,y,w,h
#     save_dict['mask'] = mask
#     # save_dict['mask_logit'] = mask_logit

#     plt.imshow(nose_image_raw)
#     mask_np = mask.astype(bool)     # ensure boolean
#     overlay = np.zeros((mask_np.shape[0], mask_np.shape[1], 4), dtype=float)
#     overlay[mask_np] = [0.0, 1.0, 0.0, 0.5]   # ONLY True pixels get color
#     plt.imshow(overlay)

#     # plt.savefig(result_image_path,bbox_inches='tight', pad_inches=2)    
#     # np.savez(mask_path,
#     #      **{k: v.cpu().numpy() if isinstance(v, torch.Tensor) else v
#     #         for k, v in save_dict.items()})
#     plt.show()
#     plt.close()

#     del inference_state
#     torch.cuda.empty_cache()

In [ ]:
processor = Sam3Processor(model, confidence_threshold=0.5)

In [ ]:
def get_normalized_bbox(bbox,image):
    width, height = image.size
    x1, y1, x2, y2 = bbox
    x,y,w,h = xyxy_to_xywh(bbox)
    box_input_xywh = torch.tensor([x,y,w,h], dtype=torch.float32).view(-1, 4)
    box_input_cxcywh = box_xywh_to_cxcywh(box_input_xywh)
    norm_box_cxcywh = normalize_bbox(box_input_cxcywh, width, height).flatten().tolist()
    return norm_box_cxcywh

In [ ]:
# col = 'right'
# other_col = 'right' if col =='left' else 'left'
# for i,line in bbox_df.iterrows():
#     image_path = os.path.join(dataset_path,line['save_path_rel'])
#     result_image_path = os.path.join(os.path.dirname(image_path), os.path.basename(image_path).split('.')[0] + '_result.jpg')

#     nose_mask_path = os.path.join(os.path.dirname(image_path), os.path.basename(image_path).split('.')[0] + '_mask.npz')
#     nostril_mask_path = os.path.join(os.path.dirname(image_path), os.path.basename(image_path).split('.')[0] + f'_{col}_nostril_mask.npz')
#     # load: state = np.load("state.npz", allow_pickle=True)
#     try:
#         image = Image.open(image_path).convert("RGB")
#     except:
#         continue


#     bbox = line['bbox']
#     nose_x1, nose_y1, nose_x2, nose_y2 = bbox
#     nose_x,nose_y,nose_w,nose_h = xyxy_to_xywh(bbox)

#     mask_dict = load_mask_dict(nose_mask_path)
#     nose_mask = mask_dict['mask']

#     nose_image_raw, nose_image_masked= extract_nose_image(image,nose_mask,nose_x1,nose_y1,nose_x2,nose_y2)
#     nose_image = nose_image_raw
#     width, height = nose_image.size

    

#     inference_state = processor.set_image(nose_image)

#     save_dict = {}

#     bbox = line[f'{col}_nostril_bbox']
#     x1, y1, x2, y2 = bbox
#     x,y,w,h = xyxy_to_xywh(bbox)
#     box_input_xywh = torch.tensor([x,y,w,h], dtype=torch.float32).view(-1, 4)
#     box_input_cxcywh = box_xywh_to_cxcywh(box_input_xywh)
#     norm_box_cxcywh = normalize_bbox(box_input_cxcywh, width, height).flatten().tolist()
    
#     processor.reset_all_prompts(inference_state)
#     # inference_state = processor.set_text_prompt(state=inference_state, prompt=f"nostril opening") 
#     inference_state = processor.add_geometric_prompt(
#         state=inference_state, box=norm_box_cxcywh, label=True
#     )

#     bbox_the_other = get_normalized_bbox(line[f'{other_col}_nostril_bbox'],nose_image)
#     inference_state = processor.add_geometric_prompt(
#         state=inference_state, box=bbox_the_other, label=False
#     )    

#     mask = inference_state['masks'].cpu().numpy()[0,0]
#     mask_logit = inference_state['masks_logits'].cpu().numpy()[0,0]
#     score = inference_state['scores'].float().cpu().numpy()[0]

#     fig, ax = plt.subplots()
#     ax.imshow(mask)
#     rect = plt.Rectangle((x1, y1), w,h, edgecolor='red', facecolor='none', linewidth=1)
#     ax.add_patch(rect)
#     plt.show()
#     plt.close()

#     save_dict['bbox_xyxy'] = x1,y1,x2,y2
#     save_dict['bbox_xywh'] = x,y,w,h
#     save_dict['mask'] = np.packbits(mask.astype(bool))
#     save_dict['shape'] = mask.shape
#     save_dict['score'] = score
#     save_dict['mask_logit'] = mask_logit.astype(np.float16)

#     np.savez_compressed(nostril_mask_path, **save_dict)

#     # --- VISUALIZATION (Object-Oriented Approach) ---

#     # Create explicit figure to avoid pyplot state leak
#     mask_npz = load_mask_dict(nostril_mask_path)
#     fig, ax = plt.subplots()
#     ax.imshow(nose_image)
    

#     x1,y1,x2,y2 = mask_npz[f'bbox_xyxy']
#     rect = plt.Rectangle((x1, y1), x2 - x1, y2 - y1, edgecolor='cyan', facecolor='none', linewidth=1)
#     ax.add_patch(rect)
#     mask = mask_npz[f'mask']
#     mask_np = mask.astype(bool)
#     overlay = np.zeros((mask_np.shape[0], mask_np.shape[1], 4), dtype=float)
#     overlay[mask_np] = [0.0, 1.0, 0.0, 0.5]
#     ax.imshow(overlay)

#     ax.set_title(f"MRN: {line['mrn']}")
#     ax.axis('off')
    
#     # Save using the figure object
#     fig.savefig(result_image_path, bbox_inches='tight', pad_inches=2)
    
#     # Explicitly close the specific figure object
#     plt.close(fig) 
    

#     # --- CLEANUP ---
#     # Explicitly delete heavy tensors
#     del inference_state
#     del mask
#     del mask_logit
    
#     # Periodically empty cache (e.g., every 10 images) rather than every image
#     # to keep speed up, unless memory is extremely tight.
#     if i % 5 == 0:
#         gc.collect()
#         torch.cuda.empty_cache()


In [ ]:
col = 'right'
other_col = 'right' if col =='left' else 'left'
for i,line in bbox_df.iterrows():
    image_path = os.path.join(dataset_path,line['save_path_rel'])
    result_image_path = os.path.join(os.path.dirname(image_path), os.path.basename(image_path).split('.')[0] + '_result.jpg')

    nose_mask_path = os.path.join(os.path.dirname(image_path), os.path.basename(image_path).split('.')[0] + '_mask.npz')
    nostril_mask_path = os.path.join(os.path.dirname(image_path), os.path.basename(image_path).split('.')[0] + f'_{col}_nostril_mask.npz')
    # load: state = np.load("state.npz", allow_pickle=True)
    try:
        image = Image.open(image_path).convert("RGB")
    except:
        continue


    bbox = line['bbox']
    nose_x1, nose_y1, nose_x2, nose_y2 = bbox
    nose_x,nose_y,nose_w,nose_h = xyxy_to_xywh(bbox)

    mask_dict = load_mask_dict(nose_mask_path)
    nose_mask = mask_dict['mask']

    nose_image_raw, nose_image_masked= extract_nose_image(image,nose_mask,nose_x1,nose_y1,nose_x2,nose_y2)
    nose_image = nose_image_raw
    width, height = nose_image.size

    

    inference_state = processor.set_image(nose_image)

    save_dict = {}

    bbox = line[f'{col}_nostril_bbox']
    x1, y1, x2, y2 = bbox
    x,y,w,h = xyxy_to_xywh(bbox)
    box_input_xywh = torch.tensor([x,y,w,h], dtype=torch.float32).view(-1, 4)
    box_input_cxcywh = box_xywh_to_cxcywh(box_input_xywh)
    norm_box_cxcywh = normalize_bbox(box_input_cxcywh, width, height).flatten().tolist()
    
    processor.reset_all_prompts(inference_state)
    # inference_state = processor.set_text_prompt(state=inference_state, prompt=f"nostril opening") 
    inference_state = processor.add_geometric_prompt(
        state=inference_state, box=norm_box_cxcywh, label=True
    )

    bbox_the_other = get_normalized_bbox(line[f'{other_col}_nostril_bbox'],nose_image)
    inference_state = processor.add_geometric_prompt(
        state=inference_state, box=bbox_the_other, label=False
    )    

    mask = inference_state['masks'].cpu().numpy()[0,0]
    mask_logit = inference_state['masks_logits'].cpu().numpy()[0,0]
    score = inference_state['scores'].float().cpu().numpy()[0]

    fig, ax = plt.subplots()
    ax.imshow(mask)
    rect = plt.Rectangle((x1, y1), w,h, edgecolor='red', facecolor='none', linewidth=1)
    ax.add_patch(rect)
    plt.show()
    plt.close()

    save_dict['bbox_xyxy'] = x1,y1,x2,y2
    save_dict['bbox_xywh'] = x,y,w,h
    save_dict['mask'] = np.packbits(mask.astype(bool))
    save_dict['shape'] = mask.shape
    save_dict['score'] = score
    save_dict['mask_logit'] = mask_logit.astype(np.float16)

    np.savez_compressed(nostril_mask_path, **save_dict)

    # --- VISUALIZATION (Object-Oriented Approach) ---

    # Create explicit figure to avoid pyplot state leak
    mask_npz = load_mask_dict(nostril_mask_path)
    fig, ax = plt.subplots()
    ax.imshow(nose_image)
    

    x1,y1,x2,y2 = mask_npz[f'bbox_xyxy']
    rect = plt.Rectangle((x1, y1), x2 - x1, y2 - y1, edgecolor='cyan', facecolor='none', linewidth=1)
    ax.add_patch(rect)
    mask = mask_npz[f'mask']
    mask_np = mask.astype(bool)
    overlay = np.zeros((mask_np.shape[0], mask_np.shape[1], 4), dtype=float)
    overlay[mask_np] = [0.0, 1.0, 0.0, 0.5]
    ax.imshow(overlay)

    ax.set_title(f"MRN: {line['mrn']}")
    ax.axis('off')
    
    # Save using the figure object
    fig.savefig(result_image_path, bbox_inches='tight', pad_inches=2)
    
    # Explicitly close the specific figure object
    plt.close(fig) 
    

    # --- CLEANUP ---
    # Explicitly delete heavy tensors
    del inference_state
    del mask
    del mask_logit
    
    # Periodically empty cache (e.g., every 10 images) rather than every image
    # to keep speed up, unless memory is extremely tight.
    if i % 5 == 0:
        gc.collect()
        torch.cuda.empty_cache()


In [ ]:
width,height